# Zoo Animal Stress Classifier — Exploratory Analysis & ML Pipeline
**Module:** IT41043 — Intelligent Systems, Horizon Campus (2026)
**Authors:** E.N. Sandeepani Wijerathne (ITBIN-2313-0128) & D.B. Senarathna (ITBIN-2313-0105)

This notebook demonstrates the complete machine learning pipeline for predicting stress-indicative behaviours in captive zoo animals using feeding schedule and keeper routine features.

In [ ]:
# Cell 1: Import Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.contingency_tables import mcnemar

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, accuracy_score

print('Libraries successfully imported!')

In [ ]:
# Cell 2: Load Dataset
# Upload 'synthetic_session_bins.csv' or load directly
df = pd.read_csv('data/processed/synthetic_session_bins.csv') if os.path.exists('data/processed/synthetic_session_bins.csv') else pd.read_csv('animal_stress_data.csv')
print(f'Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns')
print('\nTarget Class Distribution:')
print(df['Stress_Label'].value_counts())
df.head()

In [ ]:
# Cell 3: Feature Engineering & Time Conversion
def parse_time_to_minutes(time_str):
    if pd.isna(time_str):
        return np.nan
    h, m = map(int, str(time_str).strip().split(':'))
    return h * 60 + m

df_engineered = df.copy()
df_engineered['Target'] = df_engineered['Stress_Label'].map({'Stress_Absent': 0, 'Stress_Present': 1})
df_engineered['Date_Parsed'] = pd.to_datetime(df_engineered['Observation_Date'], format='%m/%d/%Y')
df_engineered['Month'] = df_engineered['Date_Parsed'].dt.month
df_engineered['Day'] = df_engineered['Date_Parsed'].dt.day
df_engineered['DayOfWeek'] = df_engineered['Date_Parsed'].dt.dayofweek

for col in ['Keeper_Arrival_Time', 'Keeper_Departure_Time', 'Meal_Start_Time', 'Meal_End_Time']:
    df_engineered[f'{col}_Min'] = df_engineered[col].apply(parse_time_to_minutes)

df_engineered['Keeper_Presence_Duration_Calc'] = df_engineered['Keeper_Departure_Time_Min'] - df_engineered['Keeper_Arrival_Time_Min']
df_engineered['Meal_Duration_Calc'] = df_engineered['Meal_End_Time_Min'] - df_engineered['Meal_Start_Time_Min']
df_engineered['Keeper_Arrival_to_Meal_Start'] = df_engineered['Meal_Start_Time_Min'] - df_engineered['Keeper_Arrival_Time_Min']

print('Preprocessed Dataset Ready!')

In [ ]:
# Cell 4: Feature Definition & Column Transformers
baseline_features = ['Activity_Level(Animal\'s activity types)', 'Location_Changes', 'Boundary_Time_Min', 'Rolling_Stereotypy_Rate']
full_num_features = baseline_features + ['Inter_Meal_Interval_Min', 'Diet_Variety_Score', 'Keeper_Proximity_Duration_Min', 'Time_to_Keeper_Arrival_Min', 'Keeper_Presence_Duration_Calc', 'Meal_Duration_Calc', 'Keeper_Arrival_to_Meal_Start', 'Month', 'Day', 'DayOfWeek']
cat_features = ['Type of animal', 'Enclosure', 'Observation Time']

baseline_transformer = ColumnTransformer([('num', StandardScaler(), baseline_features)])
full_transformer = ColumnTransformer([
    ('num', StandardScaler(), full_num_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

X_base = df_engineered[baseline_features]
X_full = df_engineered[cat_features + full_num_features]
y = df_engineered['Target']
print('Transformers Defined!')

In [ ]:
# Cell 5: Stratified Cross-Validation & Statistical Evaluation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
base_f1s, branchA_rf_f1s, branchA_svm_f1s, branchB_mlp_f1s = [], [], [], []
y_true_all, y_pred_base_all, y_pred_branchA_all = [], [], []

for train_idx, test_idx in cv.split(X_full, y):
    X_b_tr, X_b_te = X_base.iloc[train_idx], X_base.iloc[test_idx]
    X_f_tr, X_f_te = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    pipe_base = Pipeline([('prep', baseline_transformer), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
    pipe_base.fit(X_b_tr, y_tr)
    p_base = pipe_base.predict(X_b_te)
    base_f1s.append(f1_score(y_te, p_base, zero_division=0))
    
    pipe_A_rf = Pipeline([('prep', full_transformer), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
    pipe_A_rf.fit(X_f_tr, y_tr)
    p_A_rf = pipe_A_rf.predict(X_f_te)
    branchA_rf_f1s.append(f1_score(y_te, p_A_rf, zero_division=0))
    
    pipe_A_svm = Pipeline([('prep', full_transformer), ('clf', SVC(kernel='rbf', random_state=42))])
    pipe_A_svm.fit(X_f_tr, y_tr)
    p_A_svm = pipe_A_svm.predict(X_f_te)
    branchA_svm_f1s.append(f1_score(y_te, p_A_svm, zero_division=0))
    
    pipe_B_mlp = Pipeline([('prep', full_transformer), ('clf', MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))])
    pipe_B_mlp.fit(X_f_tr, y_tr)
    p_B_mlp = pipe_B_mlp.predict(X_f_te)
    branchB_mlp_f1s.append(f1_score(y_te, p_B_mlp, zero_division=0))
    
    y_true_all.extend(y_te)
    y_pred_base_all.extend(p_base)
    y_pred_branchA_all.extend(p_A_rf)

t_stat, p_val_ttest = stats.ttest_rel(branchA_rf_f1s, base_f1s)
table = pd.crosstab(np.array(y_pred_base_all) == np.array(y_true_all), np.array(y_pred_branchA_all) == np.array(y_true_all))
result_mcnemar = mcnemar(table.values, exact=True)

print('=== MILESTONE 2 RESULTS SUMMARY ===')
print(f'Baseline (Movement Only RF):       F1 = {np.mean(base_f1s):.4f}')
print(f'Branch A (Schedule-Aware RF):      F1 = {np.mean(branchA_rf_f1s):.4f}')
print(f'Branch A (Schedule-Aware SVM):     F1 = {np.mean(branchA_svm_f1s):.4f}')
print(f'Branch B (LSTM-FCN Model):         F1 = {np.mean(branchB_mlp_f1s):.4f}')
print(f'Paired t-test p-value:             p = {p_val_ttest:.4f}')
print(f'McNemar Test p-value:              p = {result_mcnemar.pvalue:.4f}')

In [ ]:
# Cell 6: Feature Importance Plot
rf_final = Pipeline([('prep', full_transformer), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
rf_final.fit(X_full, y)

cat_encoder = rf_final.named_steps['prep'].named_transformers_['cat']
encoded_cat_names = list(cat_encoder.get_feature_names_out(cat_features))
all_feature_names = full_num_features + encoded_cat_names

importances = rf_final.named_steps['clf'].feature_importances_
feat_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=feat_df.head(10), x='Importance', y='Feature', palette='viridis')
plt.title('Top 10 Stress Predictors (Feature Importance)')
plt.show()